# 10 · Reproduction check in a clean session (plan step 7, D-024)

Follows the README on a **new Colab session**, with nothing from earlier work (no Drive), and compares what it gets
with the numbers in the repository:

| Check | Passes if |
|---|---|
| splits rebuilt from the official index files | all six files byte-identical to `splits/` |
| released model (v1.0) on the test split | macro-F1 and accuracy within 0.001 (full and leaf-verified part), at most 5 of 10,709 predictions differ, same "not sure" numbers |
| demo examples | the seven photos get their expected answers |
| browser export | the ONNX model gives the Keras answers |
| one model trained again (`mobilenetv2_head`) | val macro-F1 within 0.01 of the reported 0.9158 (GPU training is not bit-deterministic, D-018) |

It also writes `requirements.lock` (repository root), the exact package versions of this environment (plan 7.2).

Re-scoring the **frozen** release model reproduces numbers that are already published; nothing is selected or changed
from it, so the test split is still used once for every decision (D-015, D-024).

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. About 25 minutes.

## 1 · Clone and install, as in the README

In [ ]:
!nvidia-smi -L || echo "No GPU: the training check needs a T4 runtime."
%cd /content
!rm -rf plant-disease-detection
!git clone -q https://github.com/AnarHummatli/plant-disease-detection.git
%cd /content/plant-disease-detection
!git log -1 --oneline
# Colab already has TensorFlow; the extras add the demo, the browser export and the tests
%pip install -q -e ".[dev,demo,web]"
!python -m plantdx.env

## 2 · Images (about 3 minutes)

In [ ]:
!python -m plantdx.data.images --out /content/data

## 3 · Reproduction check (about 20 minutes, most of it the training run)

Every line ends in `PASS`, `FAIL`, `INFO` (versions, timing) or `SKIP`. The checks run in a separate process, so the
packages installed above are used even though this notebook's kernel started before them.

In [ ]:
!python -m plantdx.reproduce --data-root /content/data --train

## 4 · Report

In [ ]:
from IPython.display import Markdown

Markdown(open("reports/reproduction/reproduction.md", encoding="utf-8").read())

## 5 · Download

`reproduction.zip` holds `reports/reproduction/reproduction.md`, `reproduction.json` and `requirements.lock`, with
their paths in the repository: send it in the chat or unpack it in the repository root and commit.

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile("/content/reproduction.zip", "w") as z:
    for name in ("reports/reproduction/reproduction.md", "reports/reproduction/reproduction.json",
                 "requirements.lock"):
        z.write(name, name)
files.download("/content/reproduction.zip")

## Done criteria (plan step 7.4)

- [ ] Cell 1 shows a Tesla T4 and the install finishes without errors
- [ ] Cell 3 ends with `0` failed lines (`... lines without failure`) and every check says `PASS`
- [ ] `reproduction.zip` is downloaded